In [ ]:
# ── Cell 1: Parameters ────────────────────────────────────────────────────────
# Edit ONLY these variables before running the notebook.

BRANCH = "main"                      # Git branch to check out
CONFIG = "configs/vhh_small.yaml"    # Config file inside the repo
USER   = "your_first_name"           # Your first name (used for logging)
RESUME = True                        # Set to True to resume from last.pt

REPO_URL = "https://github.com/ahmad04bh/Recherche_AI.git"

In [ ]:
# ── Cell 2: Setup ─────────────────────────────────────────────────────────────
# Mounts Drive, clones the repo, installs deps, copies data, verifies integrity.
# No logic lives here — all logic is in src/ (versioned in Git).

import os
from pathlib import Path

# 1. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
SHARED = Path("/content/drive/MyDrive/vhh-sh2")

# 2. Read the GitHub token from Colab Secrets (never print it)
from google.colab import userdata
GH_TOKEN = userdata.get("GH_TOKEN")

# 3. Clone / update the repo
REPO_DIR = Path("/content/Recherche_AI")
if REPO_DIR.exists():
    print("Repo already cloned — pulling latest changes …")
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    # Token is masked in all output — we redirect clone output
    MASKED_URL = f"https://***@github.com/ahmad04bh/Recherche_AI.git"
    print(f"Cloning {MASKED_URL} (branch={BRANCH}) …")
    clone_url = f"https://{GH_TOKEN}@github.com/ahmad04bh/Recherche_AI.git"
    result = !git clone --branch {BRANCH} {clone_url} {REPO_DIR} 2>&1
    # Print result with token redacted
    for line in result:
        print(line.replace(GH_TOKEN, "***") if GH_TOKEN else line)

os.chdir(REPO_DIR)

# 4. Install runtime requirements (torch is already in Colab; do NOT reinstall)
!pip install -q -r requirements.txt

# 5. Set environment variables
os.environ["PYTHONPATH"] = str(REPO_DIR)
os.environ["VHH_SHARED"] = str(SHARED)
os.environ["VHH_DATA"]   = "/content/data"
os.environ["VHH_USER"]   = USER

# 6. Copy clean data from Drive to Colab local disk (fast I/O during training)
import subprocess
data_src = str(SHARED / "data") + "/"
data_dst = "/content/data/"
Path(data_dst).mkdir(parents=True, exist_ok=True)
print(f"Syncing data {data_src} → {data_dst} …")
subprocess.run(["rsync", "-a", "--progress", data_src, data_dst], check=True)

# 7. Verify data integrity — refuses to continue if checksums differ
!python scripts/verify_data.py --data-dir real

In [ ]:
# ── Cell 3: Run ───────────────────────────────────────────────────────────────
# Launches training. src/train.py is a stub — this cell will raise
# NotImplementedError until Team B implements the training loop.

resume_flag = "--resume" if RESUME else ""
!python src/train.py --config {CONFIG} {resume_flag}